# Assignment 05 · Notebook 01: Các mô hình phát triển của CNN

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Notebook phục vụ **mục 2** của đề. Mỗi kiến trúc đánh dấu một bước phát triển của CNN, và bước
đó quy về một cơ chế viết được thành hàm. Các lớp nằm trong `a05/blocks.py`; ở đây chỉ chạy một
lượt forward trên tensor giả để kiểm tra shape và đếm tham số. **Không huấn luyện** ở mục này.

In [1]:
import sys
sys.path.insert(0, "..")
import torch
from a05 import blocks as B
from a05.export import save_metrics

torch.manual_seed(42)
x = torch.randn(2, 3, 32, 32)
print("Thiết bị: CPU · đầu vào giả", tuple(x.shape))


def probe(name, module, inp):
    out = module(inp)
    print(f"{name:20s} {str(tuple(inp.shape)):18s} -> {str(tuple(out.shape)):18s} {B.n_params(module):>10,} tham số")
    return {"params": B.n_params(module), "in": "×".join(map(str, inp.shape[1:])),
            "out": "×".join(map(str, out.shape[1:]))}

Thiết bị: CPU · đầu vào giả (2, 3, 32, 32)


## Từ LeNet đến ViT

| Kiến trúc | Năm | Cơ chế |
|---|---|---|
| LeNet-5 | 1998 | Conv → Pool → FC |
| AlexNet | 2012 | ReLU, Dropout, sâu hơn |
| VGG | 2014 | chồng conv 3×3 |
| Inception | 2014 | nhánh song song, nối kênh |
| ResNet | 2015 | $y = F(x) + x$ |
| DenseNet | 2017 | $x_l = H_l([x_0, \dots, x_{l-1}])$ |
| MobileNet | 2017 | depthwise + pointwise |
| SE-Net / CBAM | 2017–2018 | $x \odot \sigma(\mathrm{MLP}(\mathrm{GAP}(x)))$ |
| ViT | 2020 | ảnh → ô → self-attention |

In [2]:
feat16 = torch.randn(2, 16, 32, 32)
feat64 = torch.randn(2, 64, 16, 16)
res = {
    "lenet": probe("LeNet5", B.LeNet5(), x),
    "alexnet": probe("AlexNetMini", B.AlexNetMini(), x),
    "vgg": probe("VGGBlock(3→64)", B.VGGBlock(3, 64), x),
    "inception": probe("InceptionBlock", B.InceptionBlock(64, 32, 64, 16, 16), feat64),
    "residual": probe("ResidualBlock(64)", B.ResidualBlock(64), feat64),
    "dense": probe("DenseBlock(16,+12×4)", B.DenseBlock(16, 12, 4), feat16),
    "depthwise": probe("DepthwiseSep(64→128)", B.DepthwiseSeparable(64, 128), feat64),
    "se": probe("SEBlock(64)", B.SEBlock(64), feat64),
    "cbam": probe("CBAM(64)", B.CBAM(64), feat64),
}
tokens = B.PatchEmbed(3, 4, 64)(x)
res["patch"] = probe("PatchEmbed(p=4)", B.PatchEmbed(3, 4, 64), x)
res["vit"] = probe("TinyViTBlock", B.TinyViTBlock(64, 4), tokens)

LeNet5               (2, 3, 32, 32)     -> (2, 10)                62,006 tham số
AlexNetMini          (2, 3, 32, 32)     -> (2, 10)             1,575,626 tham số
VGGBlock(3→64)       (2, 3, 32, 32)     -> (2, 64, 16, 16)        38,720 tham số
InceptionBlock       (2, 64, 16, 16)    -> (2, 128, 16, 16)       27,432 tham số


ResidualBlock(64)    (2, 64, 16, 16)    -> (2, 64, 16, 16)        73,984 tham số
DenseBlock(16,+12×4) (2, 16, 32, 32)    -> (2, 64, 32, 32)        14,960 tham số
DepthwiseSep(64→128) (2, 64, 16, 16)    -> (2, 128, 16, 16)        8,768 tham số
SEBlock(64)          (2, 64, 16, 16)    -> (2, 64, 16, 16)           580 tham số
CBAM(64)             (2, 64, 16, 16)    -> (2, 64, 16, 16)           679 tham số
PatchEmbed(p=4)      (2, 3, 32, 32)     -> (2, 64, 64)             3,136 tham số
TinyViTBlock         (2, 64, 64)        -> (2, 64, 64)            33,472 tham số


## Hai phép so sánh số tham số giải thích lý do ra đời của VGG và MobileNet

**VGG:** hai conv 3×3 liên tiếp có vùng nhìn 5×5 như một conv 5×5, nhưng dùng $2 \cdot 9C^2 = 18C^2$
trọng số thay vì $25C^2$, và có thêm một hàm phi tuyến ở giữa.

**MobileNet:** conv thường $C_{in} \to C_{out}$ cần $9 C_{in} C_{out}$ trọng số; depthwise + pointwise
chỉ cần $9 C_{in} + C_{in} C_{out}$.

In [3]:
C = 64
conv5 = torch.nn.Conv2d(C, C, 5, padding=2, bias=False)
two3 = torch.nn.Sequential(torch.nn.Conv2d(C, C, 3, padding=1, bias=False), torch.nn.ReLU(),
                           torch.nn.Conv2d(C, C, 3, padding=1, bias=False))
std = torch.nn.Conv2d(64, 128, 3, padding=1, bias=False)
dws = B.DepthwiseSeparable(64, 128)
cmp = {"conv5": B.n_params(conv5), "two3": B.n_params(two3), "std": B.n_params(std), "dws": B.n_params(dws)}
cmp["vgg_saving"] = 1 - cmp["two3"] / cmp["conv5"]
cmp["dws_ratio"] = cmp["std"] / cmp["dws"]
print(f"conv 5×5: {cmp['conv5']:,} · hai conv 3×3: {cmp['two3']:,} · tiết kiệm {100 * cmp['vgg_saving']:.0f}%")
print(f"conv 3×3 64→128: {cmp['std']:,} · depthwise separable: {cmp['dws']:,} · ít hơn {cmp['dws_ratio']:.1f} lần")

conv 5×5: 102,400 · hai conv 3×3: 73,728 · tiết kiệm 28%
conv 3×3 64→128: 73,728 · depthwise separable: 8,768 · ít hơn 8.4 lần


## Kiểm tra tính chất của đường tắt và SE

Nếu nhánh $F$ của khối residual bằng 0 thì khối trở thành $\mathrm{ReLU}(x)$: tầng mới thêm vào
không thể làm mạng tệ đi so với việc bỏ qua nó. SE cho mỗi kênh một hệ số trong $(0, 1)$.

In [4]:
rb = B.ResidualBlock(64).eval()
torch.nn.init.zeros_(rb.F[4].weight)  # gamma của BN cuối = 0 thì F(x) = 0
xin = torch.relu(feat64)
identity_ok = bool(torch.allclose(rb(xin), xin))
se = B.SEBlock(64)
w = se.fc(feat64.mean(dim=(2, 3)))
print("F = 0 thì khối residual là ánh xạ đồng nhất:", identity_ok)
print(f"hệ số SE nằm trong ({w.min().item():.3f}, {w.max().item():.3f})")
_ = save_metrics("blocks", {"blocks": res, "compare": cmp, "identity_ok": identity_ok,
                            "se_min": w.min().item(), "se_max": w.max().item()})

F = 0 thì khối residual là ánh xạ đồng nhất: True
hệ số SE nằm trong (0.380, 0.625)
